In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv(r'E:\User_Retention\data\raw\orders.csv')
df.shape

(3421083, 7)

In [3]:
first_orders = df[df['order_number']==1][['user_id', 'order_dow']].copy()

In [4]:
first_orders.head()

,user_id,order_dow
0,1,2
11,2,2
26,3,1
39,4,6
45,5,3


In [5]:
first_orders.shape

(206209, 2)

In [6]:
first_orders['cohort'] = np.where(first_orders['order_dow'].isin([0, 6]), 'weekend', 'weekday')

In [7]:
df = df.merge(first_orders[['user_id', 'cohort']], on='user_id', how='left')

In [8]:
df.shape

(3421083, 8)

In [9]:
returning = df[df['order_number'] == 2][['user_id', 'cohort', 'days_since_prior_order']].copy()


returning['retention_bucket'] = pd.cut(returning['days_since_prior_order'], bins=[0, 7, 30, 60, 90, float('inf')], labels=['0-7d', '8-30d', '31-60d', '61-90d', '90d+'])


retention_dist = returning.groupby(['cohort', 'retention_bucket'])['user_id'].count().reset_index()

matrix = retention_dist.pivot(
    index='cohort',
    columns='retention_bucket',
    values='user_id'
)

matrix_pct = matrix.div(matrix.sum(axis=1), axis=0) * 100
matrix_pct.round(2)

retention_bucket,0-7d,8-30d
cohort,,
weekday,32.12,67.88
weekend,30.57,69.43
